In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import random
import time
import json
import os
import sys
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from tenacity import retry, stop_after_attempt, wait_exponential
from groq import Groq


# ========== НАЧАЛО НАСТРОЕК ==========

GROQ_API_KEYS = [
    "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",
    "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",
    "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",
    "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"
]

GROQ_MODEL = "llama-3.3-70b-versatile"

# Лимиты для одного ключа Groq (бесплатный тариф)
TOKEN_LIMIT_PER_KEY = 100000   # 100k токенов в день на ключ
REQUEST_LIMIT_PER_KEY = 10000  # 10k запросов в день на ключ

# ========== КОНЕЦ НАСТРОЕК ==========


# ========== УПРАВЛЕНИЕ API КЛЮЧАМИ ==========

class GroqAPIManager:
    """Менеджер для автоматического переключения между Groq API ключами"""

    def __init__(self, api_keys: List[str], model: str, token_limit: int, request_limit: int):
        self.keys = []
        self.current_key_index = 0
        self.total_requests = 0
        self.total_tokens = 0
        self.failed_keys = set()

        # Инициализация ключей
        for i, key in enumerate(api_keys):
            if key and key.startswith("gsk_"):
                self.keys.append({
                    "key": key,
                    "model": model,
                    "used_tokens": 0,
                    "used_requests": 0,
                    "limit_tokens": token_limit,
                    "limit_requests": request_limit,
                    "client": Groq(api_key=key)
                })
                print(f"✅ Loaded Groq key #{i+1}: {key[:20]}...")
            else:
                print(f"⚠️ Skipping invalid key #{i+1}")

        if len(self.keys) == 0:
            raise Exception("No valid Groq API keys provided!")

        print(f"\n✅ Total {len(self.keys)} Groq API keys loaded")

    def _is_rate_limit_error(self, error: Exception) -> bool:
        """Проверка, является ли ошибка rate limit"""
        error_str = str(error).lower()
        return any(x in error_str for x in ["429", "rate limit", "tokens per day", "limit reached"])

    def get_current_key(self) -> Dict:
        """Получение текущего ключа"""
        if self.current_key_index >= len(self.keys):
            raise Exception("All API keys have failed or reached limits")
        return self.keys[self.current_key_index]

    def switch_to_next_key(self) -> bool:
        """Переключение на следующий ключ. Возвращает True если успешно"""
        # Помечаем текущий ключ как проблемный
        current_key = self.keys[self.current_key_index]
        self.failed_keys.add(self.current_key_index)
        print(f"  ⚠️ Key #{self.current_key_index + 1} exhausted/failed")

        # Ищем следующий работающий ключ
        for i in range(len(self.keys)):
            if i not in self.failed_keys:
                # Проверяем лимиты
                key = self.keys[i]
                if key["used_tokens"] < key["limit_tokens"] and key["used_requests"] < key["limit_requests"]:
                    self.current_key_index = i
                    remaining_tokens = key["limit_tokens"] - key["used_tokens"]
                    remaining_requests = key["limit_requests"] - key["used_requests"]
                    print(f"  🔄 Switched to Key #{i+1}")
                    print(f"     Remaining tokens: {remaining_tokens:,}")
                    print(f"     Remaining requests: {remaining_requests:,}")
                    return True

        print("\n❌ All API keys have reached their limits!")
        return False

    def update_usage(self, tokens: int = 0, requests: int = 1):
        """Обновление статистики использования"""
        current_key = self.get_current_key()
        current_key["used_tokens"] += tokens
        current_key["used_requests"] += requests
        self.total_tokens += tokens
        self.total_requests += requests

    def get_remaining_tokens(self) -> int:
        """Оставшиеся токены на текущем ключе"""
        current_key = self.get_current_key()
        return current_key["limit_tokens"] - current_key["used_tokens"]

    def get_status(self) -> str:
        """Статус всех ключей"""
        lines = ["\n📊 Groq API Keys Status:"]
        for i, key in enumerate(self.keys):
            status = "✅" if i not in self.failed_keys else "❌"
            tokens_left = key["limit_tokens"] - key["used_tokens"]
            reqs_left = key["limit_requests"] - key["used_requests"]
            lines.append(f"  {status} Key #{i+1}: Tokens {key['used_tokens']:,}/{key['limit_tokens']:,} ({tokens_left:,} left) | Req {key['used_requests']:,}/{key['limit_requests']:,} ({reqs_left:,} left)")
        lines.append(f"\n📈 Total across all keys: {self.total_tokens:,} tokens, {self.total_requests:,} requests")
        return "\n".join(lines)

    def can_continue(self) -> bool:
        """Проверка, можно ли продолжать генерацию"""
        for i in range(len(self.keys)):
            if i not in self.failed_keys:
                key = self.keys[i]
                if key["used_tokens"] < key["limit_tokens"] and key["used_requests"] < key["limit_requests"]:
                    return True
        return False


def generate_review_with_retry(api_manager: GroqAPIManager, prompt: str, temperature: float, max_retries: int = 3) -> Tuple[Optional[str], bool]:
    """
    Генерация отзыва с автоматическим переключением между ключами
    Возвращает (review_text, success)
    """
    for attempt in range(max_retries):
        try:
            key = api_manager.get_current_key()
            client = key["client"]

            response = client.chat.completions.create(
                model=key["model"],
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=200,
                top_p=1
            )

            review_text = response.choices[0].message.content.strip()

            # Подсчёт использованных токенов
            if hasattr(response, 'usage') and response.usage:
                tokens_used = response.usage.total_tokens
            else:
                tokens_used = len(prompt.split()) + len(review_text.split()) + 50

            api_manager.update_usage(tokens=tokens_used, requests=1)
            return review_text.strip('"\''), True

        except Exception as e:
            error_msg = str(e)
            print(f"  ⚠️ Error: {error_msg[:120]}")

            if api_manager._is_rate_limit_error(e):
                if api_manager.switch_to_next_key():
                    print(f"  🔄 Retrying with next key (attempt {attempt + 1}/{max_retries})")
                    time.sleep(2)
                    continue
                else:
                    return None, False
            else:
                # Другая ошибка - пробуем с тем же ключом
                if attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 3
                    print(f"  ⏳ Waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
                    continue
                else:
                    return None, False

    return None, False


# ========== ПРОМПТЫ ДЛЯ ГЕНЕРАЦИИ ==========

CATEGORIES = [
    'Electronics_5', 'Books_5', 'Movies_and_TV_5',
    'Clothing_Shoes_and_Jewelry_5', 'Home_and_Kitchen_5',
    'Sports_and_Outdoors_5', 'Toys_and_Games_5',
    'Beauty_5', 'Pet_Supplies_5', 'Office_Products_5'
]

CATEGORY_NAMES = {
    'Electronics_5': 'electronics', 'Books_5': 'books', 'Movies_and_TV_5': 'movies or TV shows',
    'Clothing_Shoes_and_Jewelry_5': 'clothing, shoes, or jewelry', 'Home_and_Kitchen_5': 'home and kitchen products',
    'Sports_and_Outdoors_5': 'sports and outdoors equipment', 'Toys_and_Games_5': 'toys and games',
    'Beauty_5': 'beauty products', 'Pet_Supplies_5': 'pet supplies', 'Office_Products_5': 'office products'
}

PRODUCTS_BY_CATEGORY = {
    'Electronics_5': ['wireless headphones', 'smartphone', 'laptop', 'tablet', 'power bank', 'bluetooth speaker', 'smart watch'],
    'Books_5': ['thriller novel', 'cookbook', 'self-help book', 'fantasy novel', 'biography', 'science fiction book'],
    'Movies_and_TV_5': ['action movie DVD', 'documentary series', 'comedy show Blu-ray', 'drama series box set'],
    'Clothing_Shoes_and_Jewelry_5': ['running shoes', 'winter jacket', 'leather watch', 'cotton t-shirt', 'jeans', 'backpack'],
    'Home_and_Kitchen_5': ['coffee maker', 'blender', 'knife set', 'cookware set', 'cutting board', 'microwave', 'vacuum cleaner'],
    'Sports_and_Outdoors_5': ['yoga mat', 'tent', 'bike helmet', 'dumbbell set', 'fitness tracker', 'camping stove'],
    'Toys_and_Games_5': ['board game', 'action figure', 'puzzle', 'doll', 'building blocks', 'remote control car'],
    'Beauty_5': ['moisturizer', 'shampoo', 'lipstick', 'face cream', 'hair dryer', 'makeup brush set'],
    'Pet_Supplies_5': ['dog leash', 'cat bed', 'aquarium filter', 'pet food bowl', 'scratching post'],
    'Office_Products_5': ['office chair', 'desk lamp', 'mechanical pencil', 'printer paper', 'stapler', 'filing cabinet']
}

# Реальные примеры из вашего датасета
REAL_CG_EXAMPLES = [
    "Love this! Well made, sturdy, and very comfortable. I love it!Very pretty",
    "Great big numbers & easy to read, the only thing I didn't like is the size of the",
    "Does a great job of keeping the suction on. I also love that it doesn't slide.",
    "Awesome is all I have to say, the materials are good, and the quality is decent.",
    "Love this! Perfect size for an entire family!Very good quality.",
    "Fantastic, just what I wanted!!! I love the look and feel of this pillow.",
    "The only problem is that it's not really a vacuum, but it is a great product for the price!",
    "Nice, sturdy, and functional. Great for the price. Not too big or too small."
]

REAL_OR_EXAMPLES = [
    "These are just perfect, exactly what I was looking for.",
    "Great cup. Will last forever. Keeps things cool / warm.",
    "Very soft and great quality for such a low price!",
    "Works great, it's Pyrex, I'd expect nothing less from them.",
    "Just what I wanted to hang my pants and skirts.",
    "Item as described and arrived in a timely manner."
]


def get_prompt(review_type, prompt_style, product, category, rating):
    """Построение промпта в зависимости от стиля"""
    category_name = CATEGORY_NAMES.get(category, category.replace('_5', ''))

    if prompt_style == "simple":
        if review_type == "bot_generated":
            return f"""Write a short fake (bot-generated) review for {product} in {category_name} category. Rating: {rating}/5.
Make it sound like a bot: repetitive positive phrases ("great product", "highly recommend"), add an incomplete thought at the end like "Very pretty." or "I will keep my shelves in order."
Output only the review text."""

        elif review_type == "incentivized":
            return f"""Write a fake incentivized review for {product} in {category_name} category. Rating: {rating}/5.
The user got a reward. Make it unnaturally excited with exclamation marks and words like "amazing!!!", "perfect!!!", "must buy!!!".
Add a phrase like "The only problem is that it's not really a vacuum."
Output only the review text."""

        else:  # organic
            return f"""Write a short genuine review for {product} in {category_name} category. Rating: {rating}/5.
Keep it natural, mention one specific feature. No excessive excitement. Output only the review text."""

    elif prompt_style == "with_examples":
        if review_type == "organic":
            examples = REAL_OR_EXAMPLES
        else:
            examples = REAL_CG_EXAMPLES

        examples_text = "\n".join([f"- {ex}" for ex in examples[:4]])

        return f"""Here are examples of {'real' if review_type == 'organic' else 'fake'} reviews:
{examples_text}

Now generate a {review_type} review for {product} in {category_name} category. Rating: {rating}/5.
Imitate the style of the examples. Output only the review text."""

    else:  # humanized
        if review_type == "bot_generated":
            return f"""Write a fake review that sounds like a bot trying to imitate a human.
Product: {product}, category: {category_name}, rating: {rating}/5.
Include:
- one minor grammar mistake (e.g., "these done fit well", "it's more better")
- one hesitation phrase ("I'm not sure but...")
- a repetitive positive pattern ("love it, love it, love it")
- an incomplete thought at the end.
Output only the review text."""

        elif review_type == "incentivized":
            return f"""Write a review for {product} in {category_name} category. Rating: {rating}/5.
The user received a discount or gift for this review.
Make it sound like a real person but unnaturally positive.
Add:
- an exclamation ("Amazing!!")
- a weird unrelated detail ("I will keep my shelves in order")
- a phrase like "The only problem is that it's not really a vacuum."
Output only the review text."""

        else:  # organic
            return f"""Write a natural, human-sounding review for {product} in {category_name} category. Rating: {rating}/5.
Include:
- one personal experience ("I use it every morning")
- one specific like or dislike
- a conversational tone (e.g., "honestly, it's fine for the price")
No repetition, no exclamations unless justified.
Output only the review text."""


def get_rating_for_type(review_type):
    """Рейтинг в зависимости от типа отзыва"""
    if review_type == "organic":
        return random.randint(1, 5)
    elif review_type == "bot_generated":
        return random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.05, 0.2, 0.35, 0.35])[0]
    else:  # incentivized
        return random.choices([1, 2, 3, 4, 5], weights=[0.01, 0.02, 0.05, 0.15, 0.77])[0]


# ========== ОСНОВНАЯ ФУНКЦИЯ ГЕНЕРАЦИИ ==========

def generate_dataset(api_manager: GroqAPIManager, n_per_cell: int = 20):
    """Генерация датасета с автоматическим переключением ключей"""
    review_types = ["organic", "bot_generated", "incentivized"]
    prompt_styles = ["simple", "with_examples", "humanized"]
    temperatures = [0, 0.2, 0.5, 1.0]

    all_data = []
    total_combinations = len(review_types) * len(prompt_styles) * len(temperatures)
    current = 0

    # Файл для сохранения прогресса
    checkpoint_file = "generation_checkpoint.json"

    # Загружаем чекпоинт если есть
    try:
        with open(checkpoint_file, 'r') as f:
            checkpoint_data = json.load(f)
            print(f"✅ Loaded checkpoint: {len(checkpoint_data)} reviews already generated")
            all_data = checkpoint_data.get("reviews", [])
            completed_keys = checkpoint_data.get("completed_keys", [])
    except FileNotFoundError:
        checkpoint_data = {"reviews": [], "completed_keys": []}
        completed_keys = []

    for rtype in review_types:
        for pstyle in prompt_styles:
            for temp in temperatures:
                current += 1
                print(f"\n[{current}/{total_combinations}] Generating {n_per_cell} reviews for {rtype} | {pstyle} | temp={temp}")

                # Проверяем, не завершена ли уже эта комбинация
                combo_key = f"{rtype}_{pstyle}_{temp}"
                if combo_key in completed_keys:
                    print(f"  ⏭️ Skipping (already completed)")
                    continue

                for i in range(n_per_cell):
                    # Проверяем, не сгенерирован ли уже этот конкретный отзыв
                    review_key = f"{combo_key}_{i}"
                    existing = [r for r in all_data if r.get("_key") == review_key]
                    if existing:
                        print(f"  [{i+1:2d}/{n_per_cell}] Skipping (already generated)")
                        continue

                    # Проверяем, есть ли ещё токены
                    if not api_manager.can_continue():
                        print("\n❌ No more tokens available. Saving progress...")
                        save_checkpoint(checkpoint_file, all_data, completed_keys + [combo_key] if i >= n_per_cell - 1 else completed_keys)
                        return all_data

                    # Генерация
                    category = random.choice(CATEGORIES)
                    product = random.choice(PRODUCTS_BY_CATEGORY[category])
                    rating = get_rating_for_type(rtype)
                    prompt = get_prompt(rtype, pstyle, product, category, rating)

                    print(f"  Generating [{i+1:2d}/{n_per_cell}] {category} | {product} | rating={rating}")

                    review_text, success = generate_review_with_retry(api_manager, prompt, temp)

                    if not success:
                        print(f"\n❌ Generation failed. Saving checkpoint...")
                        save_checkpoint(checkpoint_file, all_data, completed_keys)
                        return all_data

                    review_data = {
                        "_key": review_key,
                        "review_text": review_text,
                        "true_label": rtype,
                        "prompt_style": pstyle,
                        "temperature": temp,
                        "category": category,
                        "product": product,
                        "rating": rating,
                        "sample_id": i + 1
                    }
                    all_data.append(review_data)

                    # Сохраняем чекпоинт каждые 10 отзывов
                    if len(all_data) % 10 == 0:
                        save_checkpoint(checkpoint_file, all_data, completed_keys)

                    print(f"      ✓ Generated (length: {len(review_text)} chars)")
                    print(f"      Remaining tokens on current key: {api_manager.get_remaining_tokens():,}")
                    time.sleep(0.3)

                # Отмечаем комбинацию как завершённую
                completed_keys.append(combo_key)
                save_checkpoint(checkpoint_file, all_data, completed_keys)

    # Удаляем чекпоинт после успешного завершения
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    return all_data


def save_checkpoint(filename, reviews, completed_keys):
    """Сохранение прогресса"""
    checkpoint = {
        "reviews": reviews,
        "completed_keys": completed_keys,
        "timestamp": datetime.now().isoformat()
    }
    with open(filename, 'w') as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)


def print_sample_reviews(df, n=10):
    """Вывод примеров сгенерированных отзывов"""
    print("\n" + "=" * 80)
    print("📝 SAMPLE GENERATED REVIEWS")
    print("=" * 80)

    for idx, row in df.head(n).iterrows():
        print(f"\n[{idx+1}] Label: {row['true_label']} | Rating: {row['rating']} | Category: {row['category']}")
        print(f"    Product: {row['product']}")
        print(f"    Style: {row['prompt_style']} | Temp: {row['temperature']}")
        print(f"    Text: {row['review_text'][:200]}..." if len(row['review_text']) > 200 else f"    Text: {row['review_text']}")
        print("-" * 80)


# ========== ЗАПУСК ==========

if __name__ == "__main__":
    print("=" * 70)
    print("🐍 FAKE REVIEW GENERATOR with Groq Multi-Key Support")
    print("=" * 70)

    # Проверяем, что есть хотя бы один ключ
    valid_keys = [k for k in GROQ_API_KEYS if k and k.startswith("gsk_")]

    if len(valid_keys) == 0:
        print("\n❌ No valid Groq API keys found!")
        print("\n📝 Instructions:")
        print("   1. Go to https://console.groq.com")
        print("   2. Sign up / Log in")
        print("   3. Go to API Keys section")
        print("   4. Create up to 4 API keys")
        print("   5. Insert them into GROQ_API_KEYS list")
        print("\nExample:")
        print('   GROQ_API_KEYS = [')
        print('       "gsk_abc123...",  # Key #1')
        print('       "gsk_def456...",  # Key #2')
        print('       "gsk_ghi789...",  # Key #3')
        print('       "gsk_jkl012...",  # Key #4')
        print('   ]')
        sys.exit(1)

    print(f"\n🔑 Loaded {len(valid_keys)} Groq API keys")
    print(f"📦 Model: {GROQ_MODEL}")
    print(f"💰 Token limit per key: {TOKEN_LIMIT_PER_KEY:,}")
    print(f"📊 Request limit per key: {REQUEST_LIMIT_PER_KEY:,}")

    print(f"\n📋 Generation config:")
    print(f"   - Review types: 3 (organic, bot_generated, incentivized)")
    print(f"   - Prompt styles: 3 (simple, with_examples, humanized)")
    print(f"   - Temperatures: 4 (0, 0.2, 0.5, 1.0)")
    print(f"   - Reviews per combination: 20")
    print(f"   - Total reviews: 720")
    print(f"   - Categories: {len(CATEGORIES)}")
    print(f"   - Products per category: ~{min(len(p) for p in PRODUCTS_BY_CATEGORY.values())}-{max(len(p) for p in PRODUCTS_BY_CATEGORY.values())}")

    print(f"\n💡 Estimated tokens per review: ~150-250")
    print(f"💡 Estimated total tokens needed: ~150,000")
    print(f"💡 With {len(valid_keys)} keys, you can generate ~{len(valid_keys) * TOKEN_LIMIT_PER_KEY:,} tokens total")

    # Создаём менеджер API
    api_manager = GroqAPIManager(
        api_keys=GROQ_API_KEYS,
        model=GROQ_MODEL,
        token_limit=TOKEN_LIMIT_PER_KEY,
        request_limit=REQUEST_LIMIT_PER_KEY
    )

    print(api_manager.get_status())

    print("\n🚀 Starting generation...")
    print("   Press Ctrl+C to stop and save progress\n")

    try:
        reviews = generate_dataset(api_manager, n_per_cell=20)

        if reviews:
            df = pd.DataFrame(reviews)
            # Убираем временные ключи перед сохранением
            if '_key' in df.columns:
                df = df.drop(columns=['_key'])

            output_file = f"synthetic_reviews_complete_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            df.to_csv(output_file, index=False, encoding='utf-8')

            print("\n" + "=" * 70)
            print(f"✅ SUCCESS! Saved {len(df)} reviews to {output_file}")
            print("=" * 70)
            print(api_manager.get_status())

            print("\n📊 Distribution by label:")
            print(df['true_label'].value_counts())

            print("\n📊 Distribution by rating per label:")
            print(pd.crosstab(df['true_label'], df['rating']))

            print_sample_reviews(df, n=10)

        else:
            print("\n❌ No reviews were generated. Check API configuration.")

    except KeyboardInterrupt:
        print("\n\n⚠️ Interrupted by user. Progress saved to generation_checkpoint.json")
        print("Run the script again to continue from where you left off.")

    print("\n✅ Done!")

🐍 FAKE REVIEW GENERATOR with Groq Multi-Key Support

🔑 Loaded 4 Groq API keys
📦 Model: llama-3.3-70b-versatile
💰 Token limit per key: 100,000
📊 Request limit per key: 10,000

📋 Generation config:
   - Review types: 3 (organic, bot_generated, incentivized)
   - Prompt styles: 3 (simple, with_examples, humanized)
   - Temperatures: 4 (0, 0.2, 0.5, 1.0)
   - Reviews per combination: 20
   - Total reviews: 720
   - Categories: 10
   - Products per category: ~4-7

💡 Estimated tokens per review: ~150-250
💡 Estimated total tokens needed: ~150,000
💡 With 4 keys, you can generate ~400,000 tokens total
✅ Loaded Groq key #1: gsk_qXPrXMVK5kkoGCrd...
✅ Loaded Groq key #2: gsk_xwI64tAM0A6Iwcdt...
✅ Loaded Groq key #3: gsk_J32MiqFuI7Km7vOf...
✅ Loaded Groq key #4: gsk_50IRBhk19KnlbmJG...

✅ Total 4 Groq API keys loaded

📊 Groq API Keys Status:
  ✅ Key #1: Tokens 0/100,000 (100,000 left) | Req 0/10,000 (10,000 left)
  ✅ Key #2: Tokens 0/100,000 (100,000 left) | Req 0/10,000 (10,000 left)
  ✅ Key #3: T